# Agent World SDK — Analysis Demo

This notebook demonstrates the **analysis modules** of the Agent World SDK:
- **Economic** — wealth distribution, Gini coefficient, inflation rate
- **Social** — network centrality, community detection, interaction patterns
- **Behavior** — survival stats, strategy classification, activity profiles
- **DataFrame** — pandas integration for all SDK data

All analysis functions are **pure computation** (no API calls). They operate on data
you've already fetched from the server.

In [ ]:
from agent_world_sdk import AgentWorldClient
from agent_world_sdk.dataframe import (
    to_dataframe,
    agents_dataframe,
    history_dataframe,
    network_dataframe,
    behavior_log_dataframe,
)

client = AgentWorldClient("http://localhost:8080", api_key="my-secret-key-123")

## 1. Fetch Raw Data

Gather agent profiles, world history, the network graph, and behavior logs.

In [ ]:
agents = client.agents.list()
profiles = [client.agents.profile(a["id"]) for a in agents]
history = client.world.history(limit=50)
network = client.export.graph(format="json")
behavior = client._get("/api/v2/export/behavior", params={"limit": 500})

print(f"Agents: {len(agents)} | History: {len(history)} ticks")
print(f"Network: {network['node_count']} nodes, {network['edge_count']} edges")
print(f"Behavior log: {behavior.get('count', 0)} entries")

## 2. Economic Analysis

Wealth distribution, Gini coefficient, and inflation tracking.

In [ ]:
# Wealth distribution across all agents
wealth = client.economic.wealth_distribution(profiles)
print("Wealth Distribution:")
print(f"  Money Gini:   {wealth['money_gini']}")
print(f"  Tokens Gini:  {wealth['tokens_gini']}")
print(f"  Top-10% money share: {wealth['top_10_money_share']}")
print(f"  Top-10% tokens share: {wealth['top_10_tokens_share']}")
print(f"  Money mean/median/std: {wealth['money_mean']} / {wealth['money_median']} / {wealth['money_std']}")
print(f"  Tokens mean/median/std: {wealth['tokens_mean']} / {wealth['tokens_median']} / {wealth['tokens_std']}")

In [ ]:
# Token trend over time
trend = client.economic.price_trend(history, field="total_tokens")
print(f"Token Trend: {trend['change_pct']:+.2f}% over {len(trend['ticks'])} ticks")
print(f"  Min: {trend['min']} | Max: {trend['max']} | Slope: {trend['slope']}")

In [ ]:
# Inflation rate (money supply growth)
inflation = client.economic.inflation_rate(history)
print(f"Cumulative inflation: {inflation['cumulative']:.2f}%")
print(f"Annualized equivalent: {inflation['annualized_equiv']:.2f}%")

## 3. Social Network Analysis

Centrality, community detection, and interaction patterns from the agent interaction graph.

In [ ]:
edges = network["edges"]
nodes = network["nodes"]

# Degree centrality
deg = client.social.degree_centrality(edges)
# Show top-5 most central agents
top5 = sorted(deg.items(), key=lambda x: x[1]["centrality"], reverse=True)[:5]
print("Top-5 agents by degree centrality:")
for aid, info in top5:
    print(f"  {aid}: degree={info['degree']}, centrality={info['centrality']}")

In [ ]:
# Betweenness centrality (bridge nodes)
bc = client.social.betweenness_centrality(edges)
top_bridge = sorted(bc.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top-5 bridge agents (betweenness centrality):")
for aid, score in top_bridge:
    print(f"  {aid}: {score}")

In [ ]:
# Community / connected-component analysis
comm = client.social.community_summary(edges, nodes)
print(f"Connected components: {comm['component_count']}")
print(f"Largest component: {comm['largest_component_size']} agents ({comm['largest_component_ratio']:.1%})")
print(f"Isolated nodes: {comm['isolated_nodes']}")

In [ ]:
# Top interactors
top_interactors = client.social.top_interactors(edges, top_n=5)
print("Most active agents (by interaction weight):")
for t in top_interactors:
    print(f"  {t['agent_id']}: weight={t['total_weight']}, partners={t['unique_partners']}")

## 4. Behavioral Pattern Analysis

Survival statistics, agent strategy classification, and activity profiles.

In [ ]:
# Survival statistics
surv = client.behavior.survival_stats(profiles)
print(f"Survival: {surv['alive_count']}/{surv['total']} alive ({surv['survival_rate']:.1%})")
print(f"Ticks survived: mean={surv['mean_ticks_survived']}, median={surv['median_ticks_survived']}, max={surv['max_ticks_survived']}")
print("\nBy phase:")
for phase, info in surv['by_phase'].items():
    print(f"  {phase}: {info['count']} agents, {info['survival_rate']:.1%} alive")

In [ ]:
# Strategy classification
entries = behavior.get("entries", [])
strat = client.behavior.strategy_classification(entries)
print("Agent strategy distribution:")
for strategy, count in strat['strategy_distribution'].items():
    print(f"  {strategy}: {count} agents")

In [ ]:
# Activity profile — top agents
activity = client.behavior.activity_profile(entries)
print("Top-10 most active agents:")
for a in activity['top_agents']:
    print(f"  {a['agent_id']}: {a['total_events']} events (dominant: {a['dominant_action']})")

In [ ]:
# Activity over time (bucketed by 10 ticks)
buckets = client.behavior.activity_over_ticks(entries, tick_bucket_size=10)
print(f"Activity buckets: {len(buckets)}")
for b in buckets[:5]:
    print(f"  Tick {b['tick_range']}: {b['count']} events (top: {b['top_event_type']})")

## 5. Pandas DataFrame Integration

Convert any SDK data to DataFrames for further analysis or visualization.

In [ ]:
# Agents to DataFrame with clean columns
df_agents = agents_dataframe(profiles)
df_agents.head()

In [ ]:
# World history to DataFrame (flattened)
df_history = history_dataframe(history)
df_history.head()

In [ ]:
# Network graph to DataFrames
df_nodes, df_edges = network_dataframe(network["nodes"], network["edges"])
print(f"Nodes: {len(df_nodes)} | Edges: {len(df_edges)}")
df_edges.head()

In [ ]:
# Behavior log to DataFrame
df_behavior = behavior_log_dataframe(entries)
df_behavior.head()

In [ ]:
# Quick visualization example
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Wealth distribution (money)
df_agents["money"].hist(ax=axes[0], bins=20)
axes[0].set_title("Money Distribution")
axes[0].set_xlabel("Money")

# Token distribution
df_agents["tokens"].hist(ax=axes[1], bins=20)
axes[1].set_title("Token Distribution")
axes[1].set_xlabel("Tokens")

# Phase counts
df_agents["phase"].value_counts().plot.bar(ax=axes[2])
axes[2].set_title("Agents by Phase")

plt.tight_layout()
plt.show()

## Summary

New analysis modules available:

| Module | Access | Functions |
|--------|--------|----------|
| **Economic** | `client.economic.*` | `wealth_distribution`, `price_trend`, `inflation_rate`, `gini`, `top_percent_share` |
| **Social** | `client.social.*` | `degree_centrality`, `betweenness_centrality`, `connected_components`, `community_summary`, `interaction_matrix`, `top_interactors` |
| **Behavior** | `client.behavior.*` | `survival_stats`, `activity_profile`, `strategy_classification`, `activity_over_ticks` |
| **DataFrame** | `from agent_world_sdk.dataframe import ...` | `to_dataframe`, `agents_dataframe`, `history_dataframe`, `network_dataframe`, `behavior_log_dataframe` |